In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
import pickle

In [34]:
data  = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [35]:
data.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='str')

In [36]:
## Processing the data
## delete RowNumber,CustomerId,Surname
data = data.drop(['RowNumber','CustomerId','Surname'],axis =1)

In [37]:
# Encode the categorical variables
label_encoder = LabelEncoder()
data['Gender'] = label_encoder.fit_transform(data['Gender'])

In [38]:
## One-Hot encode 'Geography'
onehot_encoder_geo = OneHotEncoder()
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns = onehot_encoder_geo.get_feature_names_out(['Geography']))

In [39]:
geo_encoded_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [40]:
## Combining One-Hot encode 'Geography' and original data
data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [41]:
## split the data
## Dependent feature -> EstimatedSalary
# INdependent features -> all except EstimatedSalary
X = data.drop('EstimatedSalary',axis=1)
y = data['EstimatedSalary']

In [42]:
# split the data
X_train,X_test,y_train,y_test  = train_test_split(X,y,test_size=0.20,random_state=42)

In [43]:
## Scale these Features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test) 

In [44]:
# X_train = np.asarray(X_train, dtype=np.float32)
# X_test  = np.asarray(X_test, dtype=np.float32)

# y_train = np.asarray(y_train, dtype=np.float32)
# y_test  = np.asarray(y_test, dtype=np.float32)

In [45]:
## Save the encoders and scaler for later use
with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)

with open('label_encoder.pkl','wb') as file:
    pickle.dump(label_encoder,file)

with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

## ANN Regression Problem statement

In [46]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


In [47]:
## Build the Model
model = Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),## HL1 Connected with input layer
    Dense(32,activation='relu'),## HL2
    Dense(1) # Output layer for regression ,by default activation=linear regresssion
])
 
## Compile the model
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])
model.summary()
print(model.input_shape)

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_11 (Dense)            (None, 64)                832       
                                                                 
 dense_12 (Dense)            (None, 32)                2080      
                                                                 
 dense_13 (Dense)            (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
(None, 12)


# Callbacks :-
Callbacks are functions that automatically execute during model training to monitor, control, or modify the training process.

Examples: EarlyStopping stops training when performance stops improving, and TensorBoard logs training metrics for visualization. 🚀

In [48]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime
## Setup tensor board
log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1  
)

In [49]:
## Setup tensorboard
early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [50]:
## Model Training
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[early_stopping_callback, tensorboard_callback]
)

Epoch 1/100
250/250 [==============================] - 5s 18ms/step - loss: 100423.1406 - mae: 100423.1406 - val_loss: 98695.9688 - val_mae: 98695.9688
Epoch 2/100
250/250 [==============================] - 2s 7ms/step - loss: 100341.6719 - mae: 100341.6719 - val_loss: 98552.7031 - val_mae: 98552.7031
Epoch 3/100
250/250 [==============================] - 2s 9ms/step - loss: 100155.6641 - mae: 100155.6641 - val_loss: 98316.6875 - val_mae: 98316.6875
Epoch 4/100
250/250 [==============================] - 2s 8ms/step - loss: 99886.8438 - mae: 99886.8438 - val_loss: 98010.2266 - val_mae: 98010.2266
Epoch 5/100
250/250 [==============================] - 3s 13ms/step - loss: 99562.3359 - mae: 99562.3359 - val_loss: 97659.0859 - val_mae: 97659.0859
Epoch 6/100
250/250 [==============================] - 3s 13ms/step - loss: 99216.5625 - mae: 99216.5625 - val_loss: 97315.0000 - val_mae: 97315.0000
Epoch 7/100
250/250 [==============================] - 3s 12ms/step - loss: 98869.5234 - mae: 988

In [51]:
## Load Tensorboard Extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [52]:
%tensorboard --logdir regressionlogs/fit

Reusing TensorBoard on port 6006 (pid 3818), started 0:24:32 ago. (Use '!kill 3818' to kill it.)

In [59]:
model.save('regression_model.h5')

In [54]:
## Evaluate model on the test data
test_loss,test_mae = model.evaluate(X_test,y_test)
print(f"Test MAE: {test_mae}")

63/63 [==============================] - 0s 6ms/step - loss: 50216.9258 - mae: 50216.9258
Test MAE: 50216.92578125
